# ROUGE Metrics - Review


<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/rouge-evaluation-untrained-vs-trained-llm-202504/rouge-evaluation-untrained-vs-trained-llm-202504.ipynb
</div>

The metrics that we have been using up to now with more traditional models, such as Accuracy, F1 Score or Recall, do not help us to evaluate the results of generative models.

With these models, we are beginning to use metrics such as BLEU, ROUGE, or METEOR. Metrics that are adapted to the objective of the model.

In this notebook, I want to explain how to use the ROUGE metrics to measure the quality of the summaries produced by the newest large language models.

If you're looking for more detailed explanations, you can refer to the article: [Rouge Metrics: Evaluating Summaries in Large Language Models](https://medium.com/towards-artificial-intelligence/rouge-metrics-evaluating-summaries-in-large-language-models-d200ee7ca0e6)

## Initial Setup

To install the dependencies, uncomment and run the lines below:



In [1]:
# !pip install -q rouge_score==0.1.2 evaluate==0.4.3 rouge==1.0.1 "datasets>=2.19,<3.0"

## Load the Data

In [2]:
#Import generic libraries
import numpy as np 
import pandas as pd
import torch


The dataset is available on Kaggle and comprises a collection of technological news articles compiled by MIT. The article text is located in the 'Article Body' column.

https://www.kaggle.com/datasets/deepanshudalal09/mit-ai-news-published-till-2023

In [3]:
FILE_PATH = './sample_files/article.zip'

news = pd.read_csv(FILE_PATH, index_col=0)
DOCUMENT="Article Body"

In [4]:
#Because it is just a course we select a small portion of News.
MAX_NEWS = 3
subset_news = news.head(MAX_NEWS)

In [5]:
subset_news.head()

,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url
0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...
1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...
2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...


In [6]:
articles = subset_news[DOCUMENT].tolist()

## Load the Models and create the summaries

Both models are available on Hugging Face, so we will work with the Transformers library.

We will compare two versions of the same architecture, **T5-base** (~220M parameters), so that any difference in results comes from the fine-tuning and not from the model size:

* **`t5-base`**: the original, general-purpose T5 model as released by Google. It has *not* been fine-tuned for summarization, so it only knows how to follow the generic instruction format T5 was pre-trained with (e.g. prefixing the input with a task description).
* **`flax-community/t5-base-cnn-dm`**: the same base model, but fine-tuned specifically on the **CNN/DailyMail** dataset, which is composed of news articles paired with human-written summaries. This extra training step should make it noticeably better at producing concise, relevant summaries of news text.

Comparing these two models lets us isolate the effect of fine-tuning: same architecture and size, but one has been specialized for the summarization task we're testing.

In [7]:
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name_small = "t5-base"  # 220M params
model_name_reference = "flax-community/t5-base-cnn-dm"  # Same model, fine tuned

# Another alternative:
# model_name_reference = "pszemraj/long-t5-tglobal-base-16384-booksum-V11-big_patent-V2"

/Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
#This function returns the tokenizer and the Model. 
def get_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    
    return tokenizer, model
    

In [9]:
tokenizer_small, model_small = get_model(model_name_small)

In [10]:
tokenizer_reference, model_reference = get_model(model_name_reference)

With both models downloaded and ready, we create a function that will perform the summaries.

The function takes fourth parameters:

* the list of texts to summarize.
* the tokenizer.
* the model.
* the maximum length for the generated summary

In [11]:
def create_summaries(texts_list, tokenizer, model, max_l=125):
    
    # We are going to add a prefix to each article to be summarized 
    # so that the model knows what it should do
    prefix = "Summarize this news: "  
    summaries_list = [] #Will contain all summaries

    texts_list = [prefix + text for text in texts_list]
    
    for text in texts_list:
        
        summary=""
        
        #calculate the encodings
        input_encodings = tokenizer(text, 
                                    max_length=1024, 
                                    return_tensors='pt', 
                                    padding=True, 
                                    truncation=True)

        # Generate summaries
        with torch.no_grad():
            output = model.generate(
                input_ids=input_encodings.input_ids,
                attention_mask=input_encodings.attention_mask,
                max_length=max_l,  # Set the maximum length of the generated summary
                num_beams=2,     # Set the number of beams for beam search
                early_stopping=True
            )
            
        #Decode to get the text
        summary = tokenizer.batch_decode(output, skip_special_tokens=True)
        
        #Add the summary to summaries list 
        summaries_list += summary
    return summaries_list 
    

To create the summaries, we call the 'create_summaries' function, passing both the news articles and the corresponding tokenizer and model.

In [12]:
# Creating the summaries for both models. 
summaries_small = create_summaries(articles, 
                                  tokenizer_small, 
                                  model_small)


In [13]:
summaries_reference = create_summaries(articles, 
                                      tokenizer_reference, 
                                      model_reference)

In [14]:
summaries_small

['MIT and MIT-Watson AI Lab have developed a unified framework . the system can simultaneously predict molecular properties and generate new molecules . it uses this grammar to construct viable molecules and predict their properties .',
 '\'BioAutoMATED\' is an automated machine-learning system that can select and build an appropriate model for a given dataset . it can even take care of the laborious task of data preprocessing, whittling down a months-long process to just a few hours. \'"We want to lower these barriers for a lot of folks that want to use machine learning or biology," says first co-author Jacqueline Valeri.',
 "MIT and IBM research scientists have made a computer vision model more robust by training it to work like a part of the brain that humans and other primates rely on for object recognition . 'we asked the artificial neural network to make the function of one of your inside simulated “neural” layers as similar as possible to the corresponding biological neural laye

In [15]:
summaries_reference

['Researchers created a machine-learning system that automatically learns the "language" of molecules using only a small, domain-specific dataset . The system learns to construct viable molecules and predict their properties . Computational design and Fabrication Group will be presented at the International Conference for Machine Learning .',
 "Automated machine-learning system can select and build an appropriate model for a given dataset . 'BioAutoMATED' is an automated machine-learning system . The tool includes binary classification models, multi-class classification models, and more complex neural networks .",
 "MIT and IBM researchers have found that artificial neural networks resemble the multilayered brain circuits that process visual information in humans and other primates . 'We asked it to do both of those things as well as the standard, computer vision approach,' said one expert . The network found to be more robust by training it to work like a part of the brain that humans

At first glance, it's evident that the summaries are different. 

However, it's challenging to determine which one is better. 

It's even difficult to discern whether they are significantly distinct or if there are just subtle differences between them.

This is what we are going to verify now using ROUGE. When comparing the summaries of one model with those of the other, we don't get an idea of which one is better, but rather an idea of how much the summaries have changed with the fine-tuning applied to the model.

## ROUGE

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is a family of metrics used to evaluate automatic summaries by comparing them to one or more reference (human-written) summaries.

Unlike accuracy-style metrics, ROUGE measures **n-gram overlap**: how many words or word sequences the generated summary shares with the reference. The main variants we'll see are:

* **ROUGE-1**: overlap of individual words (unigrams).
* **ROUGE-2**: overlap of consecutive word pairs (bigrams) — captures some fluency/word order.
* **ROUGE-L**: overlap based on the *Longest Common Subsequence* (LCS) between the two texts, regardless of exact word order.
* **ROUGE-Lsum**: a variant of ROUGE-L computed sentence-by-sentence and then aggregated, which tends to work better for multi-sentence summaries.

All scores range from 0 to 1 (or 0% to 100%), where higher means more overlap with the reference summary.

Let's install and load all the necessary libraries to conduct a ROUGE evaluation.

In [30]:
# `evaluate` is Hugging Face's library for loading and computing common ML/NLP metrics.
# `rouge_score` is the underlying implementation that `evaluate` uses for ROUGE.
# `sent_tokenize` (from nltk) splits text into sentences, which ROUGE-Lsum needs.
import evaluate
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer

# Downloads and instantiates the ROUGE metric, ready to `.compute(...)`
rouge_score = evaluate.load("rouge")

Calculating ROUGE is as simple as calling the *compute* function of the *rouge_score* object we created earlier. This function takes the texts to compare as arguments (`predictions` and `references`) and a third value, *use_stemmer*, which indicates whether it should reduce words to their *stem* before comparing them.

A *stemmer* reduces a word to its base form, so different inflections of the same word count as a match. For example:
* Jumping -> Jump
* Running -> Run
* Cats -> Cat

Using a stemmer (`use_stemmer=True`) makes the comparison a bit more lenient/realistic, since it won't penalize minor grammatical differences that don't affect meaning.

In [18]:
def compute_rouge_score(generated, reference):
    
    #We need to add '\n' to each line before send it to ROUGE
    generated_with_newlines = ["\n".join(sent_tokenize(s.strip())) for s in generated]
    reference_with_newlines = ["\n".join(sent_tokenize(s.strip())) for s in reference]
    
    return rouge_score.compute(
        predictions=generated_with_newlines,
        references=reference_with_newlines,
        use_stemmer=True,
        
    )

In [ ]:
import nltk
nltk.download('punkt_tab')  # Downloads the sentence-tokenizer model used by sent_tokenize()

[nltk_data] Downloading package punkt_tab to /Users/luis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# Compare the untrained model's summaries against the fine-tuned model's summaries
compute_rouge_score(summaries_small, summaries_reference)

{'rouge1': np.float64(0.47018752391886715),
 'rouge2': np.float64(0.3209013209013209),
 'rougeL': np.float64(0.34330271718331423),
 'rougeLsum': np.float64(0.44692881745120544)}

We can see that there is a difference between the two models when performing summarization. 

For example, in ROUGE-1, the similarity is 47%, while in ROUGE-2, it's a 32%. This indicates that the results are different, with some similarities but differents enough. 

However, we still don't know which model is better since we have compared them to each other and not to a reference text. But at the very least, we know that the fine-tuning process applied to the second model has significantly altered its results.

## Comparing to a Dataset with real summaries

So far we've only compared the two models' summaries **against each other**, which tells us they're different but not which one is actually better.

To answer that, we need a **ground-truth reference** written by a human. We'll use **CNN/DailyMail**, a well-known summarization dataset available through the Hugging Face **Datasets** library. Each example contains:

* `article`: the full news article.

* `highlights`: a short, human-written summary (the reference we'll compare against).By scoring each model's generated summary against these human `highlights` with ROUGE, we can see which model produces summaries closer to what a person would actually write.


In [ ]:
from datasets import load_dataset

# trust_remote_code=True is required because this dataset ships its own loading script.
cnn_dataset = load_dataset(
    "ccdv/cnn_dailymail", version="3.0.0", trust_remote_code=True

)

In [ ]:
# Reuse the same MAX_NEWS (3) articles as before, this time from the dataset's test split,
# so the comparison stays small and fast for this demo.
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS))

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 3
})

Reference summaries can vary in length. To keep the comparison fair, we compute the length of the *longest* reference `highlights` in our sample and use it as the `max_length` for both models — this way neither model is artificially cut short compared to the reference.

In [23]:
max_length = max(len(item['highlights']) for item in sample_cnn)
max_length = max_length + 10

In [24]:
summaries_t5_base = create_summaries(sample_cnn["article"], 
                                      tokenizer_small, 
                                      model_small, 
                                      max_l=max_length)

In [25]:
summaries_t5_finetuned = create_summaries(sample_cnn["article"], 
                                      tokenizer_reference, 
                                      model_reference, 
                                      max_l=max_length)

In [ ]:
# The human-written reference summaries — this is the "ground truth" both models will be scored against
real_summaries = sample_cnn['highlights']

Let's take a look at the generated summaries alongside the reference summaries provided by the dataset.

In [27]:
summaries = pd.DataFrame.from_dict(
        {
            "base": summaries_t5_base, 
            "finetuned": summaries_t5_finetuned,
            "reference": real_summaries,
        }
    )
summaries.head()

,base,finetuned,reference
0,"best died in hospice in Hickory, north Carolin...","Jimmie Best was ""the most constantly creative ...","James Best, who played the sheriff on ""The Duk..."
1,"""it doesn't matter what anyone says, he is pre...",Dr. Anthony Moschetto's attorney calls the all...,A lawyer for Dr. Anthony Moschetto says the ch...
2,president Barack Obama took part in a roundtab...,President Obama says climate change is a publi...,"""No challenge poses more of a public threat th..."


Now we can calculate the ROUGE scores for each model against the human reference summaries. Unlike the earlier comparison, this time higher scores genuinely mean "closer to how a human would summarize it".

In [28]:
compute_rouge_score(summaries_t5_base, real_summaries)

{'rouge1': np.float64(0.3050834824090638),
 'rouge2': np.float64(0.07211128178870115),
 'rougeL': np.float64(0.2095520274299344),
 'rougeLsum': np.float64(0.2662418008348241)}

In [29]:
compute_rouge_score(summaries_t5_finetuned, real_summaries)

{'rouge1': np.float64(0.31659149328289443),
 'rouge2': np.float64(0.11065084340946411),
 'rougeL': np.float64(0.2200203695620544),
 'rougeLsum': np.float64(0.24877540132887144)}

### How to interpret the ROUGE scores

Each ROUGE variant returns a value between 0 and 1 (sometimes reported as 0-100), measuring the overlap between the generated summary and the reference. A score of **1.0 (or 100)** would mean the generated summary matches the reference perfectly for that metric; **0.0** means no overlap at all.

As a rough guide for summarization tasks:

- **~0.40-0.50+ (ROUGE-1)**: Good overlap in vocabulary with the reference; the summary is covering similar content/topics.
- **~0.15-0.25 (ROUGE-2)**: Reasonable phrase-level similarity. Bigram overlap is naturally much lower than unigram overlap, even for good summaries, since it requires matching consecutive word pairs.
- **ROUGE-L / ROUGE-Lsum**: Reflect how much of the longest common subsequence is shared, regardless of exact word order — a good indicator of overall structural similarity between the texts.

So a ROUGE-1 score like **0.45** doesn't mean "45% correct"; it means 45% of the words overlap with the human reference, while the rest reflects valid differences in phrasing. There are many valid ways to summarize the same article, and ROUGE only rewards similarity to *this specific* reference, so a lower score doesn't necessarily mean a bad summary.

This is why comparing ROUGE scores alongside your own reading of the summaries is important: ROUGE gives you a quick, automatic signal, but human judgment is still needed to properly assess summary quality.
